# 02 - Improvement: train the classifier head on projected embeddings

**Idea.** In the paper the classifier is fine-tuned on raw embeddings `t`, but at test time it only ever sees reconstructed embeddings `t_hat = G(z, c)`. Those are slightly different (that gap is the small clean-accuracy drop in Table 1, e.g. 94.39 -> 94.29 on AG News). Fine-tuning just the classification layer on freshly projected training embeddings closes the mismatch, which can recover clean accuracy and make the decision boundary sit where reconstructed points actually fall.

**What changes vs the paper:** one extra, cheap training stage (only the linear head, minutes). The language model, InfoGAN, attacks and constraints are unchanged. The head is chosen by *projected validation accuracy* on the validation split (never on test data), and the original head is the fallback if nothing improves.

**Honest expectation:** this mainly helps clean accuracy and may add a little robustness. Do not expect it to move the attack columns by much on its own; it is the cheapest of the four ideas and is a good one to stack with the others.

Run notebook 00 first so the baseline results and artifacts are cached.

In [ ]:
!pip -q install -U transformers datasets accelerate textattack tensorflow_hub pandas
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # must match notebook 00
N_EVAL = 1000
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run'
HEAD_EPOCHS = 5
CLEAN_MIX = 0.0       # fraction of raw (unprojected) embeddings mixed into head training

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base')
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)

base = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, base, 'baseline', E_te, Y_te, n=N_EVAL)   # cached if notebook 00 already ran

In [ ]:
HEAD_PATH = lab._path('head_projaware.pt')
if os.path.exists(HEAD_PATH):
    head = torch.nn.Linear(768, lab.cfg['n_cls']).to(T.DEV)
    head.load_state_dict(torch.load(HEAD_PATH)); head.eval(); print('loaded head')
else:
    head = T.train_head(lab, gan, E_tr, Y_tr, E_va, Y_va, proj_mode='sample', k=BASE_K,
                        epochs=HEAD_EPOCHS, clean_mix=CLEAN_MIX)
    torch.save(head.state_dict(), HEAD_PATH)

TAG = 'projaware_head'
v = T.Victim(lab, gan=gan, head=head, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, v, TAG, E_te, Y_te, n=N_EVAL)
display(T.compare(lab, TAG, baseline_tag='baseline'))